# SESSION 2: NumPy Intermediate (Part 2: Indexing, Slicing, Broadcasting)
## Assignment Track 3: Data Science Foundations
**Case Study:** Apna Kirana Store Multi-Dimensional Inventory & Revenue Optimization Engine  
**Author:** Atharv Chaudhari (`ds-work`)

---

### Topics Covered:
1. **Indexing in 1D, 2D, and nD arrays** (`[i]`, `[i, j]`, `[i, j, k]`)
2. **Slicing arrays:** `[:]`, `[start:end]`, `[start:end:step]`
3. **Selecting rows & columns** (Vectors vs Matrices)
4. **Boolean indexing & masking** (Filtering conditions without loops)
5. **Fancy indexing** (Selecting non-contiguous rows/columns with index lists)
6. **Broadcasting rules:** Scalar-to-array & Array-to-array dimension stretching
7. **Views vs. Copies semantics:** Memory mutation mechanics and `.copy()`
8. **Practical applications:** Negative stock change rectification & Min-Max feature scaling with `keepdims=True`


---
## Task 4: Multi-Dimensional Array Generation & Subarray Extraction

**Objectives:**
- Regenerate `units_sold` (6 items $	imes$ 7 days) using `np.random.seed(42)` and `np.random.randint(5, 60, size=(6, 7))` for items:
  - `rice` (index 0), `atta` (index 1), `sugar` (index 2), `tea` (index 3), `oil` (index 4), `dal` (index 5)
- Generate `month_units` representing a 4-week sales period (4 weeks $	imes$ 6 items $	imes$ 7 days) using `np.random.randint(5, 60, size=(4, 6, 7))`.
- Extract key business slices:
  1. `tea_week`: sales for tea across the entire week (one row).
  2. `day3_sales`: sales across all items on Day 3 (one column).
  3. `staple_block`: first three items (rice, atta, sugar) across the last four days (Thursday through Sunday).
  4. `alt_days`: sales on alternate days (every second day).
  5. `dal_last`: dal sales extracted using negative indexing.
  6. `w3_tea_d6`: single scalar extraction for Week 3, Tea, Day 6 from `month_units`.
- *Hint applied:* Slices are views sharing underlying memory; use `.copy()` before editing to prevent accidental side effects.


In [1]:
import numpy as np

# Set random seed for exact reproducibility
np.random.seed(42)

# Item categories in Apna Kirana Store:
items = ['rice', 'atta', 'sugar', 'tea', 'oil', 'dal']
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# Regenerate units_sold (6 items x 7 days)
units_sold = np.random.randint(5, 60, size=(6, 7))

# Generate month_units (4 weeks x 6 items x 7 days)
month_units = np.random.randint(5, 60, size=(4, 6, 7))

print("=== units_sold (Week 1: 6 items x 7 days) ===")
print(units_sold)
print()
print("=== month_units Shape & Dimensions ===")
print("Shape:", month_units.shape, "| Ndim:", month_units.ndim, "| Total elements:", month_units.size)


=== units_sold (Week 1: 6 items x 7 days) ===
[[43 56 33 19 47 12 25]
 [43 23 27 15 15 28 57]
 [40 44 28  7 26 57  6]
 [28 48 34 42  6 25 37]
 [16 26 48 29 53 31 46]
 [32 20 19 51 55 48 59]]

=== month_units Shape & Dimensions ===
Shape: (4, 6, 7) | Ndim: 3 | Total elements: 168


In [2]:
# 1. tea_week: Row index 3 (tea) across all days
tea_week = units_sold[3, :].copy()

# 2. day3_sales: Column index 2 (Day 3: Wednesday, 0-indexed) across all items
day3_sales = units_sold[:, 2].copy()
day3_sales_idx3 = units_sold[:, 3].copy() # Day index 3 (Thursday) for explicit reference

# 3. staple_block: First 3 items (rice, atta, sugar: rows 0:3), last 4 days (Thu-Sun: cols 3:7)
staple_block = units_sold[0:3, 3:7].copy()

# 4. alt_days: Every second day using step=2 (cols 0, 2, 4, 6)
alt_days = units_sold[:, ::2].copy()

# 5. dal_last: Dal (last item) using negative index -1
dal_last = units_sold[-1, :].copy()

# 6. w3_tea_d6: Week 3 (index 2), Tea (index 3), Day 6 (index 5) in 3D month_units
w3_tea_d6 = month_units[2, 3, 5]
w3_tea_d6_raw_idx = month_units[3, 3, 6] # Direct coordinate index 3, 3, 6

print("--- Subarray Extractions ---")
print("tea_week (row index 3)           :", tea_week)
print("day3_sales (column index 2: Wed) :", day3_sales)
print("day3_sales (column index 3: Thu) :", day3_sales_idx3)
print("\nstaple_block (first 3 items, last 4 days):\n", staple_block)
print("\nalt_days (every second day - Mon, Wed, Fri, Sun):\n", alt_days)
print("\ndal_last (negative index -1)     :", dal_last)
print(f"\nw3_tea_d6 (0-indexed week 3, tea, day 6) : {w3_tea_d6}")
print(f"w3_tea_d6 (direct index [3, 3, 6])        : {w3_tea_d6_raw_idx}")


--- Subarray Extractions ---
tea_week (row index 3)           : [28 48 34 42  6 25 37]
day3_sales (column index 2: Wed) : [33 27 28 34 48 19]
day3_sales (column index 3: Thu) : [19 15  7 42 29 51]

staple_block (first 3 items, last 4 days):
 [[19 47 12 25]
 [15 15 28 57]
 [ 7 26 57  6]]

alt_days (every second day - Mon, Wed, Fri, Sun):
 [[43 33 47 25]
 [43 27 15 57]
 [40 28 26  6]
 [28 34  6 37]
 [16 48 53 46]
 [32 19 55 59]]

dal_last (negative index -1)     : [32 20 19 51 55 48 59]

w3_tea_d6 (0-indexed week 3, tea, day 6) : 5
w3_tea_d6 (direct index [3, 3, 6])        : 48


In [3]:
# Demonstration of the Hint: Slices are Views vs Copies
original_sample = np.array([10, 20, 30, 40, 50])

# Slice view
view_slice = original_sample[1:4]
view_slice[0] = 999  # Modifies underlying array

print("Hint Demonstration (View vs Copy):")
print("After modifying view_slice[0] = 999:")
print("  Original array was modified! ->", original_sample)

# Reset and use .copy()
original_sample = np.array([10, 20, 30, 40, 50])
safe_copy = original_sample[1:4].copy()
safe_copy[0] = 999

print("After modifying safe_copy[0] = 999:")
print("  Original array is protected! ->", original_sample)
print("  Isolated copy                ->", safe_copy)


Hint Demonstration (View vs Copy):
After modifying view_slice[0] = 999:
  Original array was modified! -> [ 10 999  30  40  50]
After modifying safe_copy[0] = 999:
  Original array is protected! -> [10 20 30 40 50]
  Isolated copy                -> [999  30  40]


---
## Task 1: Boolean Indexing, Fancy Indexing & Negative Masking

**Objectives:**
- Build `high_mask = units_sold > 40`.
- Print the count of `True` entries and the matching high-selling unit values.
- Use fancy indexing with `[0, 3, 5]` to extract the `rice`, `tea`, and `dal` rows into `top_items`.
- Simulate baseline stock depletion: `stock_change = units_sold - 30`.
- Use a boolean mask on a copy to set every negative value to 0 (rectifying negative stock changes).
- Print how many values changed.
- *Constraint applied:* **Do not use loops.** Pure vectorized boolean manipulation.


In [4]:
# 1. Build boolean mask for high-demand sales (> 40 units)
high_mask = units_sold > 40

# Count True occurrences using sum()
high_count = high_mask.sum()

# Extract matching values using boolean indexing
high_values = units_sold[high_mask]

print("=== High Sales Analysis (> 40 units) ===")
print("high_mask boolean matrix:")
print(high_mask)
print()
print(f"Total high-sales occurrences (True count): {high_count} out of {units_sold.size}")
print(f"Matching high-sales values: {high_values}")


=== High Sales Analysis (> 40 units) ===
high_mask boolean matrix:
[[ True  True False False  True False False]
 [ True False False False False False  True]
 [False  True False False False  True False]
 [False  True False  True False False False]
 [False False  True False  True False  True]
 [False False False  True  True  True  True]]

Total high-sales occurrences (True count): 16 out of 42
Matching high-sales values: [43 56 47 43 57 44 57 48 42 48 53 46 51 55 48 59]


In [5]:
# 2. Fancy Indexing: Select rice (index 0), tea (index 3), and dal (index 5)
top_items = units_sold[[0, 3, 5]]

print("=== Fancy Indexing: top_items (rice, tea, dal) ===")
print("Indices used: [0, 3, 5]")
print(f"top_items shape: {top_items.shape}")
print(top_items)


=== Fancy Indexing: top_items (rice, tea, dal) ===
Indices used: [0, 3, 5]
top_items shape: (3, 7)
[[43 56 33 19 47 12 25]
 [28 48 34 42  6 25 37]
 [32 20 19 51 55 48 59]]


In [6]:
# 3. Stock change simulation & negative value masking
stock_change = units_sold - 30

# Create an isolated copy to avoid mutating the original
stock_change_clean = stock_change.copy()

# Boolean mask identifying negative values
neg_mask = stock_change_clean < 0

# Count how many values are negative
num_neg_changed = neg_mask.sum()

# Vectorized replacement: set all negative entries to 0 (NO LOOPS)
stock_change_clean[neg_mask] = 0

print("=== Stock Change & Negative Value Masking ===")
print("Original stock_change (units_sold - 30):")
print(stock_change)
print()
print(f"Number of negative values rectified: {num_neg_changed}")
print()
print("Cleaned stock_change_clean (negative values clipped to 0):")
print(stock_change_clean)


=== Stock Change & Negative Value Masking ===
Original stock_change (units_sold - 30):
[[ 13  26   3 -11  17 -18  -5]
 [ 13  -7  -3 -15 -15  -2  27]
 [ 10  14  -2 -23  -4  27 -24]
 [ -2  18   4  12 -24  -5   7]
 [-14  -4  18  -1  23   1  16]
 [  2 -10 -11  21  25  18  29]]

Number of negative values rectified: 20

Cleaned stock_change_clean (negative values clipped to 0):
[[13 26  3  0 17  0  0]
 [13  0  0  0  0  0 27]
 [10 14  0  0  0 27  0]
 [ 0 18  4 12  0  0  7]
 [ 0  0 18  0 23  1 16]
 [ 2  0  0 21 25 18 29]]


---
## Task 2: Broadcasting Rules & Multi-Tier Revenue Engine

**Objectives:**
- Create `price_col` as a `(6, 1)` 2D column array from the nested list `[[52], [44], [45], [380], [140], [120]]`.
- Compute `revenue_inr = units_sold * price_col` using **array-to-array broadcasting** across columns.
- Compute `revenue_gst = revenue_inr * 1.05` using **scalar-to-array broadcasting**.
- Compute `weekend_revenue = revenue_inr * day_factor` with `day_factor = np.array([1., 1., 1., 1., 1., 1.3, 1.4])` (reflecting 30% Saturday and 40% Sunday weekend surges).
- Verify that row 1 of `revenue_inr` equals `units_sold[1] * 44` using `np.array_equal`.
- *Hint applied:* Always print `.shape` of both operands when analyzing or debugging broadcasting compatibility.


In [7]:
# 1. Construct (6, 1) price column vector from nested list
price_col = np.array([[52], [44], [45], [380], [140], [120]])

print("Operand Shapes for Broadcasting:")
print(f"  units_sold shape : {units_sold.shape} (6 rows x 7 cols)")
print(f"  price_col shape  : {price_col.shape}  (6 rows x 1 col)")

# Array-to-array broadcasting: price_col automatically stretches along axis 1 (7 cols)
revenue_inr = units_sold * price_col

print()
print("=== revenue_inr (INR): units_sold * price_col ===")
print(revenue_inr)


Operand Shapes for Broadcasting:
  units_sold shape : (6, 7) (6 rows x 7 cols)
  price_col shape  : (6, 1)  (6 rows x 1 col)

=== revenue_inr (INR): units_sold * price_col ===
[[ 2236  2912  1716   988  2444   624  1300]
 [ 1892  1012  1188   660   660  1232  2508]
 [ 1800  1980  1260   315  1170  2565   270]
 [10640 18240 12920 15960  2280  9500 14060]
 [ 2240  3640  6720  4060  7420  4340  6440]
 [ 3840  2400  2280  6120  6600  5760  7080]]


In [8]:
# 2. Scalar-to-array broadcasting: 5% GST surcharge
revenue_gst = revenue_inr * 1.05

# 3. Array-to-array broadcasting across days: Weekend surge factors
day_factor = np.array([1.0, 1.0, 1.0, 1.0, 1.0, 1.3, 1.4])

print("Broadcasting day_factor across revenue_inr:")
print(f"  revenue_inr shape : {revenue_inr.shape} (6, 7)")
print(f"  day_factor shape  : {day_factor.shape}  (7,)")
# day_factor (7,) is aligned to trailing dimension (1, 7) and broadcast down the 6 rows
weekend_revenue = revenue_inr * day_factor

print()
print("=== weekend_revenue (with 1.3x Sat & 1.4x Sun surge) ===")
print(np.round(weekend_revenue, 1))


Broadcasting day_factor across revenue_inr:
  revenue_inr shape : (6, 7) (6, 7)
  day_factor shape  : (7,)  (7,)

=== weekend_revenue (with 1.3x Sat & 1.4x Sun surge) ===
[[ 2236.   2912.   1716.    988.   2444.    811.2  1820. ]
 [ 1892.   1012.   1188.    660.    660.   1601.6  3511.2]
 [ 1800.   1980.   1260.    315.   1170.   3334.5   378. ]
 [10640.  18240.  12920.  15960.   2280.  12350.  19684. ]
 [ 2240.   3640.   6720.   4060.   7420.   5642.   9016. ]
 [ 3840.   2400.   2280.   6120.   6600.   7488.   9912. ]]


In [9]:
# 4. Verification: Check that row 1 (atta @ 44 INR) equals units_sold[1] * 44
expected_row1 = units_sold[1] * 44
actual_row1 = revenue_inr[1]

is_equal = np.array_equal(actual_row1, expected_row1)

print("=== Revenue Row 1 Verification ===")
print(f"actual_row1   : {actual_row1}")
print(f"expected_row1 : {expected_row1}")
print(f"np.array_equal check result: {is_equal}")

# Demonstration of Hint: Broadcasting Diagnostic
print("\nHint Demonstration (Broadcasting Shape Diagnostic):")
incompatible_vector = np.array([1, 2, 3, 4, 5])
try:
    _ = units_sold * incompatible_vector
except ValueError as e:
    print("Caught broadcast error successfully!")
    print(f"  units_sold shape: {units_sold.shape}")
    print(f"  incompatible_vector shape: {incompatible_vector.shape}")
    print(f"  Diagnostic: Trailing dimensions (7 vs 5) do not match and neither is 1.")


=== Revenue Row 1 Verification ===
actual_row1   : [1892 1012 1188  660  660 1232 2508]
expected_row1 : [1892 1012 1188  660  660 1232 2508]
np.array_equal check result: True

Hint Demonstration (Broadcasting Shape Diagnostic):
Caught broadcast error successfully!
  units_sold shape: (6, 7)
  incompatible_vector shape: (5,)
  Diagnostic: Trailing dimensions (7 vs 5) do not match and neither is 1.


---
## Task 3: Feature Scaling (Min-Max Normalization) with `keepdims=True`

**Objectives:**
- Scale `units_sold` item by item (row by row) into the normalized $[0, 1]$ range:
  $$\text{units\_scaled} = \frac{\text{units\_sold} - \min}{\max - \min}$$
- Subtract each row's minimum and divide by its range using `min(axis=1, keepdims=True)` and `max(axis=1, keepdims=True)`.
- Confirm that every row has minimum 0.0 and maximum 1.0 with `np.allclose`.
- *Constraints applied:* **No loops and no extra libraries.** Pure NumPy linear algebra.


In [10]:
# 1. Compute row-wise minimums and maximums preserving 2D dimensionality
row_mins = units_sold.min(axis=1, keepdims=True)
row_maxs = units_sold.max(axis=1, keepdims=True)
row_ranges = row_maxs - row_mins

print("=== Dimensionality Analysis with keepdims=True ===")
print(f"units_sold shape : {units_sold.shape} (6, 7)")
print(f"row_mins shape   : {row_mins.shape}   (6, 1) -> Preserves axis 1 for column broadcasting")
print(f"row_maxs shape   : {row_maxs.shape}   (6, 1)")
print()
print("Row minimums per item:\n", row_mins.flatten())
print("Row maximums per item:\n", row_maxs.flatten())


=== Dimensionality Analysis with keepdims=True ===
units_sold shape : (6, 7) (6, 7)
row_mins shape   : (6, 1)   (6, 1) -> Preserves axis 1 for column broadcasting
row_maxs shape   : (6, 1)   (6, 1)

Row minimums per item:
 [12 15  6  6 16 19]
Row maximums per item:
 [56 57 57 48 53 59]


In [11]:
# 2. Min-Max Normalization: Broadcast row_mins and row_ranges across axis 1
units_scaled = (units_sold - row_mins) / row_ranges

print("=== units_scaled (Row-wise Min-Max Normalized [0.0, 1.0]) ===")
print(np.round(units_scaled, 3))


=== units_scaled (Row-wise Min-Max Normalized [0.0, 1.0]) ===
[[0.705 1.    0.477 0.159 0.795 0.    0.295]
 [0.667 0.19  0.286 0.    0.    0.31  1.   ]
 [0.667 0.745 0.431 0.02  0.392 1.    0.   ]
 [0.524 1.    0.667 0.857 0.    0.452 0.738]
 [0.    0.27  0.865 0.351 1.    0.405 0.811]
 [0.325 0.025 0.    0.8   0.9   0.725 1.   ]]


In [12]:
# 3. Verification using np.allclose
scaled_mins = units_scaled.min(axis=1)
scaled_maxs = units_scaled.max(axis=1)

is_min_zero = np.allclose(scaled_mins, 0.0)
is_max_one = np.allclose(scaled_maxs, 1.0)

print("=== Normalization Verification ===")
print(f"Scaled row minimums: {scaled_mins}")
print(f"Scaled row maximums: {scaled_maxs}")
print()
print(f"np.allclose(scaled_mins, 0.0) -> {is_min_zero} (Every item has min exactly 0.0)")
print(f"np.allclose(scaled_maxs, 1.0) -> {is_max_one} (Every item has max exactly 1.0)")


=== Normalization Verification ===
Scaled row minimums: [0. 0. 0. 0. 0. 0.]
Scaled row maximums: [1. 1. 1. 1. 1. 1.]

np.allclose(scaled_mins, 0.0) -> True (Every item has min exactly 0.0)
np.allclose(scaled_maxs, 1.0) -> True (Every item has max exactly 1.0)


---
## Summary of Key Insights & Rules:

1. **Broadcasting Mechanics:**
   - Two dimensions are compatible if **(a) they are equal**, or **(b) one of them is 1**.
   - Trailing dimensions align first. A `(6, 1)` array expands to match `(6, 7)`, and a `(7,)` array expands to `(1, 7)` then broadcasts across `(6, 7)`.
2. **`keepdims=True` in Feature Scaling:**
   - Standard `min(axis=1)` collapses the dimension to `(6,)`, which cannot cleanly broadcast column-wise against `(6, 7)`.
   - `keepdims=True` retains shape `(6, 1)`, allowing immediate column-wise broadcasting without manual reshaping.
3. **Memory Safety (Views vs Copies):**
   - Slicing creates lightweight memory **views** pointing to the parent buffer. In-place edits on slices mutate the source array.
   - Using `.copy()` breaks memory linkage, creating independent buffers essential for exploratory analysis.
4. **Vectorized Logic (Zero Loops):**
   - Conditional updates (`stock_change_clean[neg_mask] = 0`) execute in compiled C memory passes, outperforming standard Python for-loops by orders of magnitude.
